In [1]:
# ViT vs CNN

# Loading ViT model
from transformers import ViTForImageClassification, ViTImageProcessor

model_name = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=10,
)

c:\Personal Projects\ViT_vs_CNN\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
# Loading Dataset
from datasets import load_dataset

dataset = load_dataset("cifar10")
#print(dataset)

# Lets start with only 1000 samples :D
train_data = dataset["train"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"]

#print(train_data)
#print(test_data)

In [3]:
# We need to preprocess the images for the ViT model
def preprocess_images(examples):
    inputs = processor(
        examples['img'],
        return_tensors="pt"
    )

    inputs['labels'] = examples['label']

    return inputs

train_data = train_data.with_transform(preprocess_images)
test_data = test_data.with_transform(preprocess_images)

In [ ]:
# Fine-tuning ViT model
from transformers import TrainingArguments, Trainer
import numpy as np
from datasets import load_metric

metric = load_metric("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, reference=labels)

training_args = TrainingArguments(
    output_dir="./vit-fine-tuned",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.756182
2,1.139400,0.612392
3,1.139400,0.312962
4,0.225100,0.268257
5,0.082300,0.257666
6,0.082300,0.245458
7,0.049200,0.246232
8,0.038100,0.247554
9,0.038100,0.248055
10,0.033400,0.248402


TrainOutput(global_step=320, training_loss=0.24692995697259904, metrics={'train_runtime': 2736.793, 'train_samples_per_second': 3.654, 'train_steps_per_second': 0.117, 'total_flos': 7.7497545904128e+17, 'train_loss': 0.24692995697259904, 'epoch': 10.0})